In [52]:
import pandas as pd
import numpy as np
from collections import defaultdict

def optimize_whole_bins(df, max_small_qty=6):
    df = df.copy()
    df['Категория'] = df['Сектор'].astype(str) + '_' + df['Группа'].astype(str)
    df['Ключ_группировки'] = df['Категория'] + '_' + df['Код'].astype(str)
    df['Количество'] = pd.to_numeric(df['Количество'], errors='coerce').fillna(0).astype(int)
    
    print(f"\n=== порог: {max_small_qty} ед. ===")

    print("\n=== АНАЛИЗ ТАР ===")
    
    bin_data = {}
    for bin_num in df['Тара'].unique():
        bin_df = df[df['Тара'] == bin_num].copy()
        bin_data[bin_num] = {
            'capacity': bin_df['Вместимость'].iloc[0],
            'total_qty': bin_df['Количество'].sum(),
            'items': bin_df.to_dict('records'),
            'codes': set(bin_df['Код'].unique()),
            'categories': set(bin_df['Категория'].unique()),
            'is_small': False,
            'target_bin': None, 
            'consolidation_group': 0
        }
        if 1 <= bin_data[bin_num]['total_qty'] <= max_small_qty:
            bin_data[bin_num]['is_small'] = True
    
    small_bins = [bin_num for bin_num, info in bin_data.items() if info['is_small']]
    print(f"Найдено малых тар (1-{max_small_qty} ед.): {len(small_bins)}")
    
    if small_bins:
        print("\nМалые тары для анализа:")
        for bin_num in small_bins[:10]:
            info = bin_data[bin_num]
            fill_percent = (info['total_qty'] / info['capacity'] * 100) if info['capacity'] > 0 else 0
            codes = list(info['codes'])
            print(f"  Тара {bin_num}: {info['total_qty']} шт, {len(codes)} кодов, заполнена на {fill_percent:.1f}%, коды: {codes[:3]}{'...'}")
    
    print("\n=== ПЛАНИРОВАНИЕ ПЕРЕМЕЩЕНИЙ ТАР ===")
    consolidation_counter = 1
    consolidation_groups = {} 
    moved_bins = 0
    skipped_bins = 0

    for bin_num in small_bins:
        if bin_data[bin_num]['target_bin'] is not None:
            continue
        
        info = bin_data[bin_num]
        total_qty = info['total_qty']
        best_target = None
        best_match_score = -1
        
        for target_bin, target_info in bin_data.items():
            if target_bin == bin_num:
                continue
            
            if target_info['target_bin'] is not None:
                continue
                
            current_load = target_info['total_qty']
            free_space = target_info['capacity'] - current_load
            if free_space < total_qty:
                continue
            

            common_codes = info['codes'] & target_info['codes']
            #приоритет: 1. общ коды 2. общ сектора 3. есть место
            if common_codes:
                match_score = 1000 + len(common_codes) * 100 + free_space
            else:
                common_categories = info['categories'] & target_info['categories']
                if common_categories:
                    match_score = 500 + len(common_categories) * 50 + free_space
                else:
                    match_score = free_space
            if match_score > best_match_score:
                best_target = target_bin
                best_match_score = match_score
                
        if best_target is not None:
            target_target = bin_data[best_target]['target_bin']
            if target_target is not None and target_target != bin_num:
                skipped_bins += 1
                continue
            
            # Регистрируем перемещение тары
            bin_data[bin_num]['target_bin'] = best_target
            
            # Обновляем информацию о целевой таре
            bin_data[best_target]['total_qty'] += total_qty
            
            # Создаем или обновляем группу объединения
            if bin_data[best_target]['consolidation_group'] == 0:
                # Создаем новую группу
                group_id = consolidation_counter
                bin_data[best_target]['consolidation_group'] = group_id
                bin_data[bin_num]['consolidation_group'] = group_id
                
                consolidation_groups[group_id] = {
                    'target_bin': best_target,
                    'source_bins': [bin_num],
                    'total_moved': total_qty
                }
                consolidation_counter += 1
            else:
                group_id = bin_data[best_target]['consolidation_group']
                bin_data[bin_num]['consolidation_group'] = group_id
                consolidation_groups[group_id]['source_bins'].append(bin_num)
                consolidation_groups[group_id]['total_moved'] += total_qty
            
            moved_bins += 1
            
            # Логируем что делаем
            common_codes = bin_data[bin_num]['codes'] & bin_data[best_target]['codes']
            common_cats = bin_data[bin_num]['categories'] & bin_data[best_target]['categories']
            
            if common_codes:
                reason = f"общие коды: {list(common_codes)[:3]}{'...' if len(common_codes) > 3 else ''}"
            elif common_cats:
                reason = f"общие категории: {list(common_cats)[:3]}{'...' if len(common_cats) > 3 else ''}"
            else:
                reason = "есть свободное место"
            
            print(f"Вся тара {bin_num} ({total_qty} шт) → тара {best_target} ({reason})")
            
        else:
            # Не нашли подходящее место
            print(f"Тара {bin_num} не перемещена: нет подходящей тары с достаточным местом")
            skipped_bins += 1
    
    print("\n=== СТАТИСТИКА ПЛАНИРОВАНИЯ ===")
    print(f"Всего малых тар: {len(small_bins)}")
    print(f"Запланировано к перемещению: {moved_bins} тар")
    print(f"Не удалось переместить: {skipped_bins} тар")
    
    if consolidation_groups:
        for group_id, group_info in consolidation_groups.items():
            target = group_info['target_bin']
            sources = group_info['source_bins']
            total_moved = group_info['total_moved']
    

    df['Целевая_тара'] = df['Тара']
    df['Группа_объединения'] = 0
    df['Нужно_переместить'] = False
    df['Тип_перемещения'] = 'Не перемещается'
    
    for idx, row in df.iterrows():
        bin_num = row['Тара']
        info = bin_data.get(bin_num, {})
        
        if info.get('target_bin') is not None:
            target_bin = info['target_bin']
            df.at[idx, 'Целевая_тара'] = target_bin
            df.at[idx, 'Группа_объединения'] = info.get('consolidation_group', 0)
            df.at[idx, 'Нужно_переместить'] = True
            target_info = bin_data.get(target_bin, {})
            common_codes = info.get('codes', set()) & target_info.get('codes', set())
            
            if common_codes:
                df.at[idx, 'Тип_перемещения'] = 'Объединение с тем же кодом'
            else:
                common_cats = info.get('categories', set()) & target_info.get('categories', set())
                if common_cats:
                    df.at[idx, 'Тип_перемещения'] = 'Объединение с той же категорией'
                else:
                    df.at[idx, 'Тип_перемещения'] = 'Объединение по свободному месту'
    

    has_cycles = False
    for bin_num, info in bin_data.items():
        target = info.get('target_bin')
        if target:
            visited = {bin_num}
            current = target
            while current and current in bin_data:
                if current in visited:
                    print(f"Обнаружен цикл: тара {bin_num} → {current}")
                    has_cycles = True
                    break
                visited.add(current)
                current = bin_data[current].get('target_bin')
    if not has_cycles:
        print("Циклов не обнаружено")
    
    capacity_issues = 0
    for bin_num in df['Целевая_тара'].unique():
        bin_df = df[df['Целевая_тара'] == bin_num]
        total_qty = bin_df['Количество'].sum()
        
        if len(bin_df) > 0:
            capacity = bin_df['Вместимость'].iloc[0]
            capacity
            if total_qty > capacity:
                capacity_issues += 1
    
    if capacity_issues == 0:
        print("  ✅ Вместимость не превышена")
    
    # Освобожденные тары
    freed_bins = set()
    for bin_num, info in bin_data.items():
        if info.get('target_bin') is not None:
            # Проверяем, что в таре больше нет товаров после перемещения
            if df[df['Тара'] == bin_num]['Нужно_переместить'].all():
                freed_bins.add(bin_num)
    
    print(f"\nОсвобождено тар: {len(freed_bins)}")
    
    df = df.drop(['Категория', 'Ключ_группировки'], axis=1, errors='ignore')
    
    moves_stats = {
        'перемещено_тар': moved_bins,
        'не_перемещено_тар': skipped_bins,
        'создано_групп': len(consolidation_groups)
    }
    
    return df, freed_bins, moves_stats, consolidation_groups

def create_consolidation_report(df_result, consolidation_groups):
    if not consolidation_groups:
        return None
    
    report = []
    for group_id, group_info in consolidation_groups.items():
        target_bin = group_info['target_bin']
        source_bins = group_info['source_bins']
        
        # Собираем информацию о группе
        group_df = df_result[df_result['Группа_объединения'] == group_id]
        
        if len(group_df) > 0:
            # Разделяем на исходные и целевую тары
            source_df = group_df[group_df['Тара'].isin(source_bins)]
            target_df = group_df[group_df['Целевая_тара'] == target_bin]
            
            # Информация о товарах
            total_qty = group_df['Количество'].sum()
            moved_qty = source_df['Количество'].sum()
            
            # Анализ кодов
            source_codes = set(source_df['Код'].unique())
            target_codes = set(target_df['Код'].unique())
            common_codes = source_codes & target_codes
            
            # Категории
            source_cats = set(source_df['Сектор'] + ' / ' + source_df['Группа'])
            target_cats = set(target_df['Сектор'] + ' / ' + target_df['Группа'])
            common_cats = source_cats & target_cats
            
            report.append({
                'Группа': group_id,
                'Целевая_тара': target_bin,
                'Перемещаемые_тары': ', '.join(map(str, source_bins)),
                'Всего_тары': ', '.join(map(str, sorted(set(source_bins + [target_bin])))),
                'Товара_перемещено': moved_qty,
                'Всего_в_целевой': total_qty,
                'Общих_кодов': len(common_codes),
                'Общих_категорий': len(common_cats),
                'Операций': len(source_bins)  # Каждая тара - одна операция
            })
    
    return pd.DataFrame(report)

def analyze_results(df_result, freed_bins):
    """Анализ результатов оптимизации"""
    print("\n=== АНАЛИЗ РЕЗУЛЬТАТОВ ===")
    
    # 1. Общая статистика
    total_records = len(df_result)
    moved_records = df_result['Нужно_переместить'].sum()
    
    print(f"Всего записей: {total_records}")
    print(f"Записей для перемещения: {moved_records} ({moved_records/total_records*100:.1f}%)")
    
    # 2. Статистика по тарам
    original_bins = df_result['Тара'].nunique()
    target_bins = df_result['Целевая_тара'].nunique()
    
    print(f"\nТар было: {original_bins}")
    print(f"Тар стало: {target_bins}")
    print(f"Освобождено тар: {len(freed_bins)} ({len(freed_bins)/original_bins*100:.1f}% от исходных)")
    
    # 3. Анализ групп объединения
    consolidation_groups = df_result['Группа_объединения'].unique()
    consolidation_groups = [g for g in consolidation_groups if g > 0]
    
    if consolidation_groups:
        print(f"\nГрупп объединения: {len(consolidation_groups)}")
        
        # Анализируем каждую группу
        groups_info = []
        for group_id in consolidation_groups:
            group_df = df_result[df_result['Группа_объединения'] == group_id]
            source_bins = group_df['Тара'].unique()
            target_bin = group_df['Целевая_тара'].iloc[0]
            
            groups_info.append({
                'group': group_id,
                'source_bins': len(source_bins),
                'target_bin': target_bin,
                'total_qty': group_df['Количество'].sum()
            })
        
        # Сортируем по количеству объединенных тар
        groups_info.sort(key=lambda x: x['source_bins'], reverse=True)
        
        print("\nКрупнейшие объединения:")
        for info in groups_info[:5]:
            print(f"  Группа {info['group']}: {info['source_bins']} тар → тара {info['target_bin']} ({info['total_qty']} шт)")
    
    # 4. Типы перемещений
    if 'Тип_перемещения' in df_result.columns:
        move_types = df_result[df_result['Нужно_переместить']]['Тип_перемещения'].value_counts()
        print(f"\nТипы перемещений:")
        for move_type, count in move_types.items():
            percent = count / moved_records * 100 if moved_records > 0 else 0
            print(f"  {move_type}: {count} записей ({percent:.1f}%)")


# Настройки
MAX_SMALL_QTY = 6  # Освобождаем тары с 1-6 единицами

# 1. Чтение файла
input_file = "C:/Users/k.belkova/forpy/shlop.xlsx"
output_file = f"C:/Users/k.belkova/forpy/shlop_цельные_тары_{MAX_SMALL_QTY}_ед.xlsx"

try:
    print(f"Чтение файла: {input_file}")
    df = pd.read_excel(input_file)
    df_original = df.copy()
    print(f"Файл прочитан. Записей: {len(df)}, Тар: {df['Тара'].nunique()}")
    
    # Анализ до оптимизации
    print("\n=== СОСТОЯНИЕ ДО ОПТИМИЗАЦИИ ===")
    
    # Анализ по тарам
    bin_stats = df.groupby('Тара').agg({
        'Количество': 'sum',
        'Код': 'nunique',
        'Вместимость': 'first'
    }).reset_index()
    
    # Малые тары
    small_bins = bin_stats[
        (bin_stats['Количество'] >= 1) & 
        (bin_stats['Количество'] <= MAX_SMALL_QTY)
    ]
    
    print(f"Тар с 1-{MAX_SMALL_QTY} единицами: {len(small_bins)}")
    
    if len(small_bins) > 0:
        print("\nПримеры малых тар:")
        for _, row in small_bins.head(5).iterrows():
            bin_df = df[df['Тара'] == row['Тара']]
            categories = (bin_df['Сектор'] + '/' + bin_df['Группа']).unique()
            print(f"  Тара {row['Тара']}: {row['Количество']} шт, {row['Код']} кодов, {len(categories)} категорий")
    
except Exception as e:
    print(f"Ошибка чтения файла: {e}")



# 3. Сохраняем результат
try:
    df_result.to_excel(output_file, index=False)
    print(f"\nОсновной результат сохранен: {output_file}")
except Exception as e:
    print(f"Ошибка при сохранении: {e}")


consolidation_report = create_consolidation_report(df_result, consolidation_groups)
if consolidation_report is not None:
    print("\nГруппы объединения (цельные тары):")
    print(consolidation_report.to_string(index=False))
else:
    print("Нет объединений тар")

# 5. Анализируем результаты
analyze_results(df_result, freed_bins)


def create_exact_format_view(df_result):
    
    # Берем только группы объединения
    df_groups = df_result[df_result['Группа_объединения'] > 0].copy()
    if len(df_groups) == 0:
        return pd.DataFrame()
    target_bins_by_group = {}
    
    for group_id in df_groups['Группа_объединения'].unique():
        group_df = df_groups[df_groups['Группа_объединения'] == group_id]
        
        # Находим все уникальные 'Целевая_тара' в группе
        target_candidates = group_df['Целевая_тара'].unique()
        
        # Ищем тару, которая является целью для других, но сама не является целью
        potential_targets = []
        
        for bin_num in target_candidates:
            # Проверяем, перемещается ли эта тара куда-то
            bin_is_moving = False
            for _, row in group_df.iterrows():
                if row['Тара'] == bin_num and row['Нужно_переместить']:
                    bin_is_moving = True
                    break
            
            # Если тара НЕ перемещается, она может быть целевой
            if not bin_is_moving:
                potential_targets.append(bin_num)
        
        if potential_targets:
            target_bins_by_group[group_id] = potential_targets[0]
        else:
            if len(target_candidates) > 0:
                target_bins_by_group[group_id] = target_candidates[0]
            else:
                target_bins_by_group[group_id] = None
    # 2. Создаем отчет
    report_rows = []
    
    for group_id in sorted(df_groups['Группа_объединения'].unique()):
        group_df = df_groups[df_groups['Группа_объединения'] == group_id]
        target_bin = target_bins_by_group.get(group_id)

        if target_bin:          
            # Товары уже в целевой таре
            existing_in_target = df_original[df_original['Тара'] == target_bin]['Количество'].sum()
            # Товары, которые перемещаются в целевую тару
            moving_to_target = group_df[
                (group_df['Целевая_тара'] == target_bin) & 
                (group_df['Нужно_переместить'])
            ]['Количество'].sum()
            
            total_in_target = existing_in_target + moving_to_target
        else:
            total_in_target = 0
        
        # 1. Строка с заголовком группы (итого для целевой тары)
        report_rows.append({
            'Признак объединения': group_id,
            'Адрес': '',
            'Тара': '',
            'Сумма по Количеству': total_in_target,
            'Куда': ''
        })
        
        # 2. Строки с тарами (все тары в группе)
        # Сначала целевая тара (если есть)
        if target_bin:
            target_bin_qty = df_original[df_original['Тара'] == target_bin]['Количество'].sum()
            adres_list = df_original[df_original['Тара'] == target_bin]['Адрес'].tolist()
            report_rows.append({
                'Признак объединения': group_id,
                'Адрес': adres_list[0],
                'Тара': target_bin,
                'Сумма по Количеству': target_bin_qty,
                'Куда': 'да'
            })
        
        # Потом остальные тары (которые перемещаются в целевую)
        # Сортируем по номеру тары
        other_bins = sorted([b for b in group_df['Тара'].unique() if b != target_bin])
        other_bins
        for bin_num in other_bins:
            bin_qty = group_df[group_df['Тара'] == bin_num]['Количество'].sum()
            adres_list = df_original[df_original['Тара'] == bin_num]['Адрес'].tolist()
            report_rows.append({
                'Признак объединения': group_id,
                'Адрес': adres_list[0],
                'Тара': bin_num,
                'Сумма по Количеству': bin_qty,
                'Куда': ''
            })
    
    report_df = pd.DataFrame(report_rows)
    report_df['Признак объединения'] = report_df['Признак объединения'].astype(int)
    return report_df


df_result, freed_bins, moves_stats, consolidation_groups = optimize_whole_bins(df, MAX_SMALL_QTY)
report = create_exact_format_view(df_result)

output_path = "C:/Users/k.belkova/forpy/отчет_объединения_исправленный.xlsx"
report.to_excel(output_path, index=False)
print(f"\n✅ Отчет сохранен: {output_path}")

Чтение файла: C:/Users/k.belkova/forpy/shlop.xlsx
Файл прочитан. Записей: 4419, Тар: 423

=== СОСТОЯНИЕ ДО ОПТИМИЗАЦИИ ===
Тар с 1-6 единицами: 34

Примеры малых тар:
  Тара 386.0: 2.0 шт, 1.0 кодов, 1 категорий
  Тара 1671.0: 2.0 шт, 2.0 кодов, 2 категорий
  Тара 5913.0: 6.0 шт, 4.0 кодов, 2 категорий
  Тара 6491.0: 5.0 шт, 3.0 кодов, 2 категорий
  Тара 6947.0: 6.0 шт, 6.0 кодов, 3 категорий

Основной результат сохранен: C:/Users/k.belkova/forpy/shlop_цельные_тары_6_ед.xlsx

Группы объединения (цельные тары):
 Группа  Целевая_тара                           Перемещаемые_тары                                          Всего_тары  Товара_перемещено  Всего_в_целевой  Общих_кодов  Общих_категорий  Операций
      1        195890                                         386                                         386, 195890                  2                2            1                1         1
      2        201673                                        1671                               